# Сохранение данных парсинга

После того как мы успешно выполнили HTTP-запрос (библиотекой `requests` или аналогичной) и извлекли необходимые данные из HTML-кода, перед нами встает задача - сохранить эти данные для дальнейшего анализа или использования.

Наиболее популярные форматы для хранения структурированных данных - это `CSV` (табличное представление) и `JSON` (иерархическое представление).

## 1. Модуль `csv` (стандартная библиотека)

Модуль `csv` предоставляет классы для чтения и записи табличных данных в формат `CSV` (Comma-Separated Values). Он встроен в Python, поэтому не требует установки.

### 1.1. Запись простого CSV-файла

Для записи нам понадобится создать объект `writer` и передать в него строки с данными.

**Синтаксис ключевых функций**:

* `open(file, mode, encoding, newline)` - стандартная функция для открытия файла.
    - `file`: имя файла.
    - `mode`: режим (`'w'` - запись (перезапись), `'a'` - добавление).
    - `encoding`: кодировка (рекомендуется `'utf-8'`, а `'utf-8-sig'` - если файл предназначен для windows-приложений).
    - `newline=''`: обязательный параметр при работе с `csv` на Windows, чтобы избежать добавление лишних пустых строк в `csv`-файл.

* `csv.writer(fileobj, delimiter=',')` - создает объект для записи.
    - `fileobj`: открытый файловый объект.
    - `delimiter`: символ-разделитель (по умолчанию `','`).
    - `lineterminator`: завершение строки (по умолчанию `'\r\n'`)
    - `quotechar`: символ кавычек (по умолчанию `'"'`)/`"'"`
    - `quoting`: режим кавычек:
        * `csv.QUOTE_MINIMAL` - только когда необходимо (по умолчанию).
        * `csv.QUOTE_ALL` - все поля в кавычки.
        * `csv.QUOTE_NONNUMERIC` - все нечисловые поля в кавычки.
        * `csv.QUOTE_NONE` - никогда не ставить кавычки.

* `writer.writerow(sequence)` - записывает одну строку (принимает список, кортеж).
* `writer.writerows(list_of_sequences)` - записывает несколько строк (принимает список списков).

In [2]:
import csv

def save_to_csv_simple(data, filename):
    """Сохраняет данные в CSV, получая список списков."""
    if not data:
        print("Нет данных для сохранения.")
        return

    # Получаем заголовки из ключей первого элемента
    headers = data[0].keys()

    with open(filename, 'w', encoding='utf-8', newline='') as csvfile:
        writer = csv.writer(csvfile, delimiter=';')  # Используем ';' для совместимости с Excel

        # Записываем заголовки
        writer.writerow(headers)

        # Записываем данные
        for item in data:
            # Преобразуем значения словаря в список
            row = [str(value) for value in item.values()]  # Преобразуем всё в строку для безопасности
            writer.writerow(row)

    print(f"Данные успешно сохранены в {filename}")

# Использование
# save_to_csv_simple(scraped_data, 'books_simple.csv')

In [ ]:
import requests
from bs4 import BeautifulSoup
# import csv

url = 'https://parsinger.ru/html/index4_page_1.html'
base_url = 'https://parsinger.ru/html/'

# функция для получения BeautifulSoup-объекта, принимает ссылку на страницу
def make_soup(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, 'lxml')
    return soup

# формируем ссылки на все доступные страницы
pages = (f"{base_url}{a.get('href')}" for a in make_soup(url).select_one('div.pagen').select('a'))
# список списков для данных о товарах
data = []

with requests.Session() as session:
    for page in pages:
        soup = make_soup(page)
        # берем все карточки товаров
        divs = soup.select('div.item')
        # обрабатываем каждую карточку товара
        for div in divs:
            # список для данных об одном товаре
            product_data = []
            
            name = div.select_one('a.name_item').text.strip()
            price = div.select_one('p.price').text.strip()
            # отдельно обрабатываем составное описание товара
            description = [li.text.split(': ')[1].strip() for li in div.select_one('div.description').select('li')]

            # заносим данные о товаре в общий список
            product_data.extend((name, *description, price))
            data.append(product_data)

# записываем данные о товарах в csv-файл
try:
    with open('result.csv', mode='w', encoding='utf-8', newline='') as file:
        writer = csv.writer(file, delimiter=';')
        header = ['Наименование', 'Бренд', 'Форм-фактор', 'Ёмкость', 'Объем буферной памяти', 'Цена']
        writer.writerow(header)
        for row in data:
            writer.writerow(row)
    print("✅ Данные успешно записаны в файл")
except Exception as e:
    print(f"❌ Ошибка при записи в файл: {e}")

✅ Данные успешно записаны в файл


### 1.2. Запись с помощью `DictWriter` (Рекомендуемый способ)

`DictWriter` более удобен, так как позволяет работать напрямую со словарями, сопоставляя ключи словаря с заголовками колонок.

**Синтаксис**:

* `csv.DictWriter(fileobj, fieldnames, delimiter)` - создает объект для записи словарей.
    - `fieldnames`: последовательность (список, кортеж) ключей, определяющих порядок колонок.
* `writer.writeheader()` - записывает строку заголовков на основе `fieldnames`.
* `writer.writerow(dictionary)` - записывает одну строку из словаря.
* `writer.writerows(list_of_dicts)` - записывает несколько строк из списка словарей.

In [5]:
def save_to_csv_dict(data, filename):
    """Сохраняет данные в CSV, используя DictWriter."""
    if not data:
        print("Нет данных для сохранения.")
        return

    fieldnames = data[0].keys()

    with open(filename, 'w', encoding='utf-8', newline='') as csvfile:
        writer = csv.DictWriter(
            csvfile,
            fieldnames=fieldnames,
            delimiter=',',  # Можно поставить ';'
            quoting=csv.QUOTE_MINIMAL  # Добавляет кавычки только при необходимости
        )

        # Записываем заголовки
        writer.writeheader()

        # Записываем все строки
        writer.writerows(data)

    print(f"Данные успешно сохранены в {filename}")

# Использование
# save_to_csv_dict(scraped_data, 'books_dict.csv')

## 2. Библиотека `pandas`

`pandas` - это библиотека для анализа данных. Она предоставляет высокоуровневые структуры данных (`DataFrame`) и функции для работы с ними. Если вы планируете в дальнейшем анализировать данные, pandas - лучший выбор.

### 2.1. Преобразование в `DataFrame` и сохранение в `CSV`

**Синтаксис ключевых методов**:

* `pd.DataFrame(data)` - создает объект `DataFrame`.
    - `data`: может быть списком словарей, словарем списков и т.д.

* `DataFrame.to_csv(filename, index, encoding, sep)` - сохраняет `DataFrame` в `CSV`.
    - `index`: записывать ли индекс строки (по умолчанию `True`).
    - `encoding`: кодировка.
    - `sep`: разделитель (по умолчанию `','`).

In [ ]:
import pandas as pd

def save_to_csv_pandas(data, filename):
    """Сохраняет данные в CSV с помощью pandas."""
    if not data:
        print("Нет данных для сохранения.")
        return

    # 1. Создаем DataFrame
    df = pd.DataFrame(data)
    print("DataFrame создан:")
    print(df.head())  # Выводим первые строки для проверки

    # 2. Сохраняем в CSV
    df.to_csv(filename, index=False, encoding='utf-8', sep=',')
    print(f"Данные успешно сохранены в {filename}")

# Использование
# save_to_csv_pandas(scraped_data, 'books_pandas.csv')

## 3. Сохранение в формат `JSON`

`JSON` (JavaScript Object Notation) - это легкий, текстовый формат обмена данными, основанный на синтаксисе объектов JavaScript.

**Главная задача JSON**: передавать структурированные данные между сервером и веб-приложением (или между разными сервисами) в удобочитаемом виде.

`JSON` строится на двух универсальных структурах данных, которые есть в любом языке:

1.  **Объект (Dictionary / Hash Map):** Неупорядоченный набор пар `Ключ: Значение`.
2.  **Массив (Array / List):** Упорядоченный список значений.

**Строгие правила синтаксиса:**

*   **Ключи объекта** всегда заключаются в **двойные кавычки** (`"name"`). Одинарные кавычки (`'name'`) недопустимы.
*   Данные разделяются запятыми.
*   Фигурные скобки `{}` содержат объект.
*   Квадратные скобки `[]` содержат массив.

В `JSON` ключ (**Key** - имя свойства) всегда должен быть **строкой** (`string`), всегда в **двойных кавычках**, желательно без пробелов (для ключей рекомендуется использовать `camelCase` или `snake_case`).

Значением (**Value**) может быть:
*   **Строка:** В двойных кавычках (`"Hello"`).
*   **Число:** Целое или с плавающей точкой (`42`, `3.14`).
*   **Булево:** `true` или `false`.
*   **Null:** Пустое значение (`null`).
*   **Объект:** Другой вложенный объект `{...}`.
*   **Массив:** `[...]`.


```json
// пример описания книги в формате json
{
  "title": "Чистый код",
  "author": "Роберт Мартин",
  "year": 2013,
  "isPublished": true,
  "tags": ["programming", "best practices", "software engineering"],
  "publisher": {
    "name": "Питер",
    "country": "Russia"
  },
  "reviews": null
}
```

### 3.1. Использование модуля `json` (стандартная библиотека)

`JSON` входит в стандартный пакет Python, устанавливать его не нужно, достаточно просто импортировать  `import json`. Модуль `json` предоставляет простые функции для сериализации (преобразования Python-объекта в `JSON`-строку) и десериализации (преобразования `JSON`-строк в Python-объекты).

**Синтаксис ключевых функций**:

* `json.dump(obj, fileobj, ensure_ascii, indent)` - записывает Python-объект в **файл** в формате `JSON`.
    - `obj`: объект для сохранения (список, словарь).
    - `fileobj`: открытый файловый объект.
    - `ensure_ascii`: если `False`, то символы не-ASCII (например, кириллица) будут записаны "как есть", а не в виде кодов `\uXXXX`. Важно для русского языка.
    - `indent`: количество пробелов для отступов, чтобы файл был читаемым (pretty-printing).

* `json.dumps(obj)` - то же, что и `dump`, но возвращает **строку**, а не записывает в файл.

In [7]:
import json

def save_to_json(data, filename):
    """Сохраняет данные в JSON-файл."""
    if not data:
        print("Нет данных для сохранения.")
        return

    with open(filename, 'w', encoding='utf-8') as jsonfile:
        json.dump(
            data,
            jsonfile,
            ensure_ascii=False,  # Для корректного отображения кириллицы
            indent=4  # Для отступов
        )

    print(f"Данные успешно сохранены в {filename}")

# Использование
# save_to_json(scraped_data, 'books.json')

In [ ]:
import requests
from bs4 import BeautifulSoup
import json

url = 'https://parsinger.ru/html/index4_page_1.html'
base_url = 'https://parsinger.ru/html/'

# функция для получения BeautifulSoup-объекта, принимает ссылку на страницу
def make_soup(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, 'lxml')
    return soup

# формируем ссылки на все доступные страницы

# ссылки на страницы категорий
cat_pages = (f"{base_url}{a.get('href')}" for a in make_soup(url).select_one('div.nav_menu').select('a'))
# итоговый список всех ссылок
links = []
# для каждой категории формируем список ссылок на доступные страницы
for page in cat_pages:
    soup = make_soup(page)
    cat_links = (f"{base_url}{a.get('href')}" for a in soup.select_one('div.pagen').select('a'))
    links.extend(cat_links)

# список словарей для данных о товарах
data = []


with requests.Session() as session:
    for link in links:
        soup = make_soup(link)
        # берем все карточки товаров
        divs = soup.select('div.item')
        # обрабатываем каждую карточку товара
        for div in divs:
            name = div.select_one('a.name_item').text.strip()
            price = div.select_one('p.price').text.strip()
            # отдельно обрабатываем составное описание товара
            description = [li.text.split(': ') for li in div.select_one('div.description').select('li')]
            # удаляем лишние пробелы в характеристиках и значениях
            description = [[key.strip(), value.strip()] for key, value in description]
            
            # заносим данные о товаре в виде словаря в общий список
            data.append(dict((['Наименование', name], *description, ['Цена', price])))


# записываем данные о товарах в json-файл
try:
    with open('result.json', mode='w', encoding='utf-8', newline='') as file:
        json.dump(data, file, indent=4, ensure_ascii=False)
    print("✅ Данные успешно записаны в файл")
except Exception as e:
    print(f"❌ Ошибка при записи в файл: {e}")

✅ Данные успешно записаны в файл


In [1]:
import requests
from bs4 import BeautifulSoup
import json

url = 'https://parsinger.ru/html/index2_page_1.html'
base_url = 'https://parsinger.ru/html/'

# функция для получения BeautifulSoup-объекта, принимает ссылку на страницу
def make_soup(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, 'lxml')
    return soup

# формируем ссылки на все доступные страницы товаров

# ссылки на страницы с категорией товаров
pages = (f"{base_url}{a.get('href')}" for a in make_soup(url).select_one('div.pagen').select('a'))
# итоговый список всех ссылок
links = []
# для каждой страницы формируем список ссылок на страницы товаров
for page in pages:
    soup = make_soup(page)
    page_links = (f"{base_url}{a.get('href')}" for a in soup.select('a.name_item'))
    links.extend(page_links)

# список словарей для данных о товарах
data = []

with requests.Session() as session:
    # обрабатываем каждую страницу товара
    for link in links:
        soup = make_soup(link)

        name = soup.select_one('p#p_header').text.strip()
        article = soup.select_one('p.article').text.split(': ')[1].strip()
        
        # отдельно обрабатываем составное описание товара, формируем отдельный словарь
        description = {li.get('id'): li.text.split(': ')[1].strip() for li in soup.select_one('ul#description').select('li')}
        
        count = soup.select_one('span#in_stock').text.split(': ')[1].strip()
        price = soup.select_one('span#price').text.strip()
        old_price = soup.select_one('span#old_price').text.strip()

        # заносим данные о товаре в виде словаря в общий список
        data.append(dict((
            ['categories', 'mobile'],
            ['name', name],
            ['article', article],
            ['description', description],
            ['count', count],
            ['price', price],
            ['old_price', old_price],
            ['link', link]
        )))

# записываем данные о товарах в json-файл
try:
    with open('result.json', mode='w', encoding='utf-8', newline='') as file:
        json.dump(data, file, indent=4, ensure_ascii=False)
    print("✅ Данные успешно записаны в файл")
except Exception as e:
    print(f"❌ Ошибка при записи в файл: {e}")

✅ Данные успешно записаны в файл


### 3.2. Проблема сериализации нестандартных типов

При сохранении данных, полученных из `BeautifulSoup`, или просто нестандартных типов (например, `Decimal` для цен или `datetime`), мы можем столкнуться с ошибкой:
`TypeError: Object of type 'Something' is not JSON serializable`

**Решение**: Нужно преобразовать данные в базовые типы (строки, числа, списки, словари).

**Пример обработки "грязных" данных**:
Представим, что библиотека `bs4` вернула нам объекты `Tag`, а цена пришла как строка с символом валюты.

In [8]:
# Пример "сырых" данных, которые мы можем получить от bs4
raw_items = [
    {'title': '<h1>Изучаем Python</h1>', 'price': '3 500 руб.', 'in_stock': 'В наличии'},
    {'title': '<h1>Python. К вершинам</h1>', 'price': '4 200 руб.', 'in_stock': 'Нет'}
]

def clean_data_for_json(raw_list):
    """Очищает и преобразует данные для JSON."""
    cleaned_data = []
    for item in raw_list:
        cleaned_item = {}
        for key, value in item.items():
            # 1. Избавляемся от HTML-тегов (очень грубый пример)
            if key == 'title' and isinstance(value, str):
                cleaned_item[key] = value.replace('<h1>', '').replace('</h1>', '')
            # 2. Чистим цену (убираем пробелы и 'руб.', преобразуем в int)
            elif key == 'price' and isinstance(value, str):
                cleaned_item[key] = int(value.replace(' ', '').replace('руб.', ''))
            # 3. Булево значение
            elif key == 'in_stock':
                cleaned_item[key] = (value == 'В наличии')
            else:
                cleaned_item[key] = value
        cleaned_data.append(cleaned_item)
    return cleaned_data

# Сохраняем очищенные данные
# cleaned_scraped_data = clean_data_for_json(raw_items)
# save_to_json(cleaned_scraped_data, 'books_cleaned.json')